In [1]:
def extract_par_utterances(filepath):
    utterances = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line.startswith('*PAR:'):
                try:
                    _, utterance = line.split(':', 1)
                    utterances.append(utterance.strip())
                except ValueError:
                    continue  # skip malformed lines
    return utterances

import re

def extract_utterance_with_times(utt):
    # Extract timestamp and remove from text
    match = re.search(r'\x15(\d+)_(\d+)\x15', utt)
    if match:
        start_time = int(match.group(1))
        end_time = int(match.group(2))
        utt = re.sub(r'\x15\d+_\d+\x15', '', utt).strip()
    else:
        return
    return {
        "utterance": utt.strip(),
        "start_time": start_time,
        "end_time": end_time
    }

In [ ]:
import os
dementia_dir = "./Pitt/Dementia/cookie"
control_dir = "./Pitt/Control/cookie"

data = []
labels = []
for file in os.listdir(dementia_dir):
    if '.cha' not in file:
        continue
    session = []
    for u in extract_par_utterances(os.path.join(dementia_dir, file)):
        utter = extract_utterance_with_times(u)
        if utter:
            session.append(utter)
    data.append(session)
    labels.append(1)
for file in os.listdir(control_dir):
    if '.cha' not in file:
        continue
    session = []
    for u in extract_par_utterances(os.path.join(control_dir, file)):
        utter = extract_utterance_with_times(u)
        if utter:
            session.append(utter)
    data.append(session)
    labels.append(0)

In [3]:
import numpy as np
import pandas as pd
import re
from typing import List, Dict, Tuple
import pickle

# PyTorch libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

# Word2Vec and other utilities
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt

class AlzheimersDataset(Dataset):
    """Custom PyTorch Dataset for Alzheimer's detection with sentence-level data"""
    
    def __init__(self, sessions_data, labels):
        """
        Args:
            sessions_data: List of sessions, where each session is a list of sentences,
                          and each sentence is a list of word vectors
            labels: List of labels (0 or 1)
        """
        self.sessions_data = sessions_data
        self.labels = labels
    
    def __len__(self):
        return len(self.sessions_data)
    
    def __getitem__(self, idx):
        session = self.sessions_data[idx]
        label = self.labels[idx]
        
        # Convert to tensors
        # session: list of sentences, each sentence is a numpy array of word vectors
        session_tensors = []
        sentence_lengths = []
        
        for sentence in session:
            if len(sentence) > 0:
                sentence_tensor = torch.FloatTensor(sentence)
                session_tensors.append(sentence_tensor)
                sentence_lengths.append(len(sentence))
            else:
                # Handle empty sentences
                sentence_tensor = torch.zeros(1, sentence.shape[1] if len(sentence.shape) > 1 else 100)
                session_tensors.append(sentence_tensor)
                sentence_lengths.append(1)
        
        return session_tensors, sentence_lengths, torch.LongTensor([label])

def collate_hierarchical_batch(batch):
    """Custom collate function for hierarchical data"""
    sessions = []  # List of sessions
    all_sentence_lengths = []  # List of sentence lengths for each session
    session_lengths = []  # Number of sentences per session
    labels = []
    
    for session_tensors, sentence_lengths, label in batch:
        sessions.append(session_tensors)
        all_sentence_lengths.append(sentence_lengths)
        session_lengths.append(len(session_tensors))
        labels.append(label)
    
    # Pad sentences within each session to the same length
    max_sentence_length = max([max(lengths) for lengths in all_sentence_lengths])
    max_session_length = max(session_lengths)
    
    # Get embedding dimension from first sentence
    embed_dim = sessions[0][0].shape[1]
    
    # Create padded batch tensor
    batch_tensor = torch.zeros(len(batch), max_session_length, max_sentence_length, embed_dim)
    batch_sentence_lengths = torch.zeros(len(batch), max_session_length, dtype=torch.long)
    batch_session_lengths = torch.LongTensor(session_lengths)
    
    for i, (session, sent_lengths) in enumerate(zip(sessions, all_sentence_lengths)):
        for j, (sentence, sent_len) in enumerate(zip(session, sent_lengths)):
            actual_len = min(sent_len, max_sentence_length)
            if actual_len > 0:
                batch_tensor[i, j, :actual_len, :] = sentence[:actual_len]
                batch_sentence_lengths[i, j] = actual_len
    
    labels = torch.cat(labels, dim=0)
    
    return batch_tensor, batch_sentence_lengths, batch_session_lengths, labels

class SentenceLSTM(nn.Module):
    """LSTM to encode individual sentences"""
    
    def __init__(self, input_size, hidden_size, dropout=0.3):
        super(SentenceLSTM, self).__init__()
        self.hidden_size = hidden_size
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=2,  # Add multiple layers for dropout to work
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, word_vectors, lengths):
        """
        Args:
            word_vectors: (batch_size, max_sentence_length, input_size)
            lengths: (batch_size,) actual lengths of sentences
        Returns:
            sentence_representation: (batch_size, hidden_size * 2)
        """
        # Pack sequences
        packed_input = pack_padded_sequence(
            word_vectors, lengths.cpu(), 
            batch_first=True, enforce_sorted=False
        )
        
        # LSTM forward pass
        packed_output, (hidden, cell) = self.lstm(packed_input)
        
        # Use final hidden states from both directions
        # hidden: (2, batch_size, hidden_size) for bidirectional
        sentence_repr = torch.cat((hidden[-2], hidden[-1]), dim=1)  # Concatenate forward and backward
        
        return self.dropout(sentence_repr)

class SessionLSTM(nn.Module):
    """LSTM to encode sequences of sentences (sessions)"""
    
    def __init__(self, input_size, hidden_size, dropout=0.3):
        super(SessionLSTM, self).__init__()
        self.hidden_size = hidden_size
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=2,  # Add multiple layers for dropout to work
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, sentence_representations, lengths):
        """
        Args:
            sentence_representations: (batch_size, max_session_length, sentence_hidden_size * 2)
            lengths: (batch_size,) actual number of sentences per session
        Returns:
            session_representation: (batch_size, hidden_size * 2)
        """
        # Pack sequences
        packed_input = pack_padded_sequence(
            sentence_representations, lengths.cpu(),
            batch_first=True, enforce_sorted=False
        )
        
        # LSTM forward pass
        packed_output, (hidden, cell) = self.lstm(packed_input)
        
        # Use final hidden states from both directions
        session_repr = torch.cat((hidden[-2], hidden[-1]), dim=1)
        
        return self.dropout(session_repr)

class HierarchicalAlzheimersLSTM(nn.Module):
    """Hierarchical LSTM: Word -> Sentence -> Session -> Classification"""
    
    def __init__(self, word_embed_size, sentence_hidden_size, session_hidden_size, 
                 mlp_hidden_size, num_classes=2, dropout=0.3):
        super(HierarchicalAlzheimersLSTM, self).__init__()
        
        self.word_embed_size = word_embed_size
        self.sentence_hidden_size = sentence_hidden_size
        self.session_hidden_size = session_hidden_size
        
        # Sentence-level LSTM
        self.sentence_lstm = SentenceLSTM(
            input_size=word_embed_size,
            hidden_size=sentence_hidden_size,
            dropout=dropout
        )
        
        # Session-level LSTM
        self.session_lstm = SessionLSTM(
            input_size=sentence_hidden_size * 2,  # *2 for bidirectional
            hidden_size=session_hidden_size,
            dropout=dropout
        )
        
        # Classification MLP with more regularization
        self.classifier = nn.Sequential(
            nn.Linear(session_hidden_size * 2, mlp_hidden_size),  # *2 for bidirectional
            nn.BatchNorm1d(mlp_hidden_size),  # Add batch normalization
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden_size, mlp_hidden_size // 2),
            nn.BatchNorm1d(mlp_hidden_size // 2),  # Add batch normalization
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden_size // 2, num_classes)
        )
        
    def forward(self, batch_tensor, sentence_lengths, session_lengths):
        """
        Args:
            batch_tensor: (batch_size, max_session_length, max_sentence_length, word_embed_size)
            sentence_lengths: (batch_size, max_session_length)
            session_lengths: (batch_size,)
        """
        batch_size, max_session_length, max_sentence_length, word_embed_size = batch_tensor.shape
        
        # Reshape for sentence-level processing
        # (batch_size * max_session_length, max_sentence_length, word_embed_size)
        reshaped_tensor = batch_tensor.view(-1, max_sentence_length, word_embed_size)
        reshaped_lengths = sentence_lengths.view(-1)
        
        # Filter out zero-length sentences
        valid_mask = reshaped_lengths > 0
        if valid_mask.sum() == 0:
            # Handle edge case where all sentences are empty
            sentence_representations = torch.zeros(
                batch_size * max_session_length, 
                self.sentence_hidden_size * 2,
                device=batch_tensor.device
            )
        else:
            valid_sentences = reshaped_tensor[valid_mask]
            valid_lengths = reshaped_lengths[valid_mask]
            
            # Process valid sentences through sentence LSTM
            valid_sentence_repr = self.sentence_lstm(valid_sentences, valid_lengths)
            
            # Create full sentence representations tensor
            sentence_representations = torch.zeros(
                batch_size * max_session_length,
                self.sentence_hidden_size * 2,
                device=batch_tensor.device
            )
            sentence_representations[valid_mask] = valid_sentence_repr
        
        # Reshape back to session format
        # (batch_size, max_session_length, sentence_hidden_size * 2)
        session_input = sentence_representations.view(
            batch_size, max_session_length, self.sentence_hidden_size * 2
        )
        
        # Process through session LSTM
        session_representations = self.session_lstm(session_input, session_lengths)
        
        # Final classification
        output = self.classifier(session_representations)
        
        return output

class HierarchicalAlzheimersDetectionPipeline:
    def __init__(self, 
                 word2vec_size=100, 
                 sentence_lstm_units=64,
                 session_lstm_units=64,
                 mlp_hidden_units=32,
                 max_sentence_length=None,
                 max_session_length=None,
                 device=None):
        """
        Initialize the hierarchical Alzheimer's detection pipeline
        """
        self.word2vec_size = word2vec_size
        self.sentence_lstm_units = sentence_lstm_units
        self.session_lstm_units = session_lstm_units
        self.mlp_hidden_units = mlp_hidden_units
        self.max_sentence_length = max_sentence_length
        self.max_session_length = max_session_length
        
        # Set device
        if device is None:
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        else:
            self.device = device
            
        print(f"Using device: {self.device}")
        
        # Initialize components
        self.word2vec_model = None
        self.model = None
        self.vocab = set()
        
    def preprocess_text(self, text: str) -> List[str]:
        """Clean and tokenize text"""
        # Clean up extra spaces and punctuation
        text = re.sub(r'[^\w\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        text = text.lower().strip()
        
        # Tokenize
        tokens = text.split()
        return [token for token in tokens if len(token) > 1]  # Filter out single characters
    
    def extract_sentences_hierarchical(self, data: List[List[Dict]]) -> Tuple[List[List[List[str]]], List[int]]:
        """Extract sentences in hierarchical format: sessions -> sentences -> words"""
        all_sentences_flat = []  # For Word2Vec training
        hierarchical_data = []   # sessions -> sentences -> words
        
        for session in data:
            session_sentences = []
            for utterance_dict in session:
                utterance = utterance_dict['utterance']
                tokens = self.preprocess_text(utterance)
                if tokens:  # Only add non-empty sentences
                    session_sentences.append(tokens)
                    all_sentences_flat.append(tokens)
            
            if session_sentences:
                hierarchical_data.append(session_sentences)
            else:
                # Handle empty sessions
                hierarchical_data.append([['empty']])
                all_sentences_flat.append(['empty'])
        
        return hierarchical_data, all_sentences_flat
    
    def train_word2vec(self, sentences: List[List[str]]):
        """Train Word2Vec model on the sentences"""
        print("Training Word2Vec model...")
        
        self.word2vec_model = Word2Vec(
            sentences=sentences,
            vector_size=self.word2vec_size,
            window=15,
            min_count=4,
            workers=4,
            epochs=100
        )
        
        # Build vocabulary
        self.vocab = set(self.word2vec_model.wv.key_to_index.keys())
        print(f"Word2Vec vocabulary size: {len(self.vocab)}")
    
    def sentence_to_vectors(self, sentence: List[str]) -> np.ndarray:
        """Convert a sentence to a sequence of word vectors"""
        vectors = []
        for word in sentence:
            if word in self.vocab:
                vectors.append(self.word2vec_model.wv[word])
            else:
                # Use zero vector for unknown words
                vectors.append(np.zeros(self.word2vec_size))
        
        if vectors:
            return np.array(vectors)
        else:
            return np.zeros((1, self.word2vec_size))
    
    def prepare_hierarchical_sequences(self, hierarchical_data: List[List[List[str]]]) -> List[List[np.ndarray]]:
        """Convert hierarchical data to sequences of word vectors"""
        sessions_data = []
        
        for session in hierarchical_data:
            session_vectors = []
            for sentence in session:
                sentence_vectors = self.sentence_to_vectors(sentence)
                session_vectors.append(sentence_vectors)
            sessions_data.append(session_vectors)
        
        return sessions_data
    
    def fit(self, data: List[List[Dict]], labels: List[int], 
            validation_split=0.2, batch_size=16, epochs=50, learning_rate=0.001):
        """Train the complete hierarchical pipeline"""
        
        print("Starting hierarchical pipeline training...")
        
        # Step 1: Extract hierarchical sentences and train Word2Vec
        hierarchical_data, all_sentences_flat = self.extract_sentences_hierarchical(data)
        self.train_word2vec(all_sentences_flat)
        
        # Step 2: Convert to hierarchical sequences
        print("Converting to hierarchical sequences...")
        sessions_data = self.prepare_hierarchical_sequences(hierarchical_data)
        
        print(f"Number of sessions: {len(sessions_data)}")
        print(f"Average sentences per session: {np.mean([len(session) for session in sessions_data]):.1f}")
        print(f"Average words per sentence: {np.mean([len(sentence) for session in sessions_data for sentence in session]):.1f}")
        
        # Step 3: Split data
        X_train, X_val, y_train, y_val = train_test_split(
            sessions_data, labels, 
            test_size=validation_split, random_state=42, stratify=labels
        )
        
        # Step 4: Create datasets and dataloaders
        train_dataset = AlzheimersDataset(X_train, y_train)
        val_dataset = AlzheimersDataset(X_val, y_val)
        
        train_loader = DataLoader(
            train_dataset, batch_size=batch_size, shuffle=True,
            collate_fn=collate_hierarchical_batch
        )
        val_loader = DataLoader(
            val_dataset, batch_size=batch_size, shuffle=False,
            collate_fn=collate_hierarchical_batch
        )
        
        # Step 5: Initialize model
        self.model = HierarchicalAlzheimersLSTM(
            word_embed_size=self.word2vec_size,
            sentence_hidden_size=self.sentence_lstm_units,
            session_hidden_size=self.session_lstm_units,
            mlp_hidden_size=self.mlp_hidden_units,
            num_classes=2,
            dropout=0.5  # Increase dropout
        ).to(self.device)
        
        # Step 6: Setup training with better regularization
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.model.parameters(), lr=learning_rate, weight_decay=1e-4)  # Add L2 regularization
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
        
        # Early stopping parameters
        patience = 7
        best_val_acc = 0
        patience_counter = 0
        
        # Training loop
        train_losses = []
        val_losses = []
        train_accs = []
        val_accs = []
        
        print("Starting training with early stopping...")
        
        for epoch in range(epochs):
            # Training phase
            self.model.train()
            train_loss = 0
            train_correct = 0
            train_total = 0
            
            for batch_tensor, sentence_lengths, session_lengths, batch_y in train_loader:
                batch_tensor = batch_tensor.to(self.device)
                sentence_lengths = sentence_lengths.to(self.device)
                session_lengths = session_lengths.to(self.device)
                batch_y = batch_y.squeeze().to(self.device)
                
                optimizer.zero_grad()
                outputs = self.model(batch_tensor, sentence_lengths, session_lengths)
                loss = criterion(outputs, batch_y)
                loss.backward()
                
                # Gradient clipping - more aggressive
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=0.5)
                
                optimizer.step()
                
                train_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                train_total += batch_y.size(0)
                train_correct += (predicted == batch_y).sum().item()
            
            # Validation phase
            self.model.eval()
            val_loss = 0
            val_correct = 0
            val_total = 0
            
            with torch.no_grad():
                for batch_tensor, sentence_lengths, session_lengths, batch_y in val_loader:
                    batch_tensor = batch_tensor.to(self.device)
                    sentence_lengths = sentence_lengths.to(self.device)
                    session_lengths = session_lengths.to(self.device)
                    batch_y = batch_y.squeeze().to(self.device)
                    
                    outputs = self.model(batch_tensor, sentence_lengths, session_lengths)
                    loss = criterion(outputs, batch_y)
                    
                    val_loss += loss.item()
                    _, predicted = torch.max(outputs.data, 1)
                    val_total += batch_y.size(0)
                    val_correct += (predicted == batch_y).sum().item()
            
            # Calculate metrics
            train_acc = 100 * train_correct / train_total
            val_acc = 100 * val_correct / val_total
            avg_train_loss = train_loss / len(train_loader)
            avg_val_loss = val_loss / len(val_loader)
            
            train_losses.append(avg_train_loss)
            val_losses.append(avg_val_loss)
            train_accs.append(train_acc)
            val_accs.append(val_acc)
            
            scheduler.step(avg_val_loss)
            
            # Early stopping logic
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0
                torch.save(self.model.state_dict(), 'best_hierarchical_alzheimers_model.pth')
                print(f"New best validation accuracy: {best_val_acc:.2f}% (Epoch {epoch+1})")
            else:
                patience_counter += 1
            
            if epoch % 5 == 0 or patience_counter == 0:
                print(f'Epoch [{epoch+1}/{epochs}]')
                print(f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%')
                print(f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%')
                print(f'Patience: {patience_counter}/{patience}')
                print('-' * 50)
            
            # Early stopping
            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs")
                break
        
        # Load best model
        self.model.load_state_dict(torch.load('best_hierarchical_alzheimers_model.pth'))
        
        print(f"Training completed! Best validation accuracy: {best_val_acc:.2f}%")
        
        return {
            'train_losses': train_losses,
            'val_losses': val_losses,
            'train_accs': train_accs,
            'val_accs': val_accs,
            'best_val_acc': best_val_acc
        }
    
    def predict(self, data: List[List[Dict]]) -> np.ndarray:
        """Make predictions on new data"""
        if self.model is None or self.word2vec_model is None:
            raise ValueError("Model not trained yet. Call fit() first.")
        
        # Prepare hierarchical sequences
        hierarchical_data, _ = self.extract_sentences_hierarchical(data)
        sessions_data = self.prepare_hierarchical_sequences(hierarchical_data)
        
        # Create dataset and dataloader
        dataset = AlzheimersDataset(sessions_data, [0] * len(sessions_data))  # Dummy labels
        dataloader = DataLoader(
            dataset, batch_size=16, shuffle=False,
            collate_fn=collate_hierarchical_batch
        )
        
        # Make predictions
        self.model.eval()
        all_predictions = []
        all_probabilities = []
        
        with torch.no_grad():
            for batch_tensor, sentence_lengths, session_lengths, _ in dataloader:
                batch_tensor = batch_tensor.to(self.device)
                sentence_lengths = sentence_lengths.to(self.device)
                session_lengths = session_lengths.to(self.device)
                
                outputs = self.model(batch_tensor, sentence_lengths, session_lengths)
                probabilities = torch.softmax(outputs, dim=1)
                predictions = torch.argmax(outputs, dim=1)
                
                all_predictions.extend(predictions.cpu().numpy())
                all_probabilities.extend(probabilities.cpu().numpy())
        
        return np.array(all_predictions), np.array(all_probabilities)
    
    def evaluate(self, data: List[List[Dict]], labels: List[int]):
        """Evaluate the model on test data"""
        predictions, probabilities = self.predict(data)
        
        accuracy = accuracy_score(labels, predictions)
        print(f"Test Accuracy: {accuracy:.4f}")
        print("\nClassification Report:")
        
        return accuracy, predictions, probabilities

In [4]:
from sklearn.model_selection import StratifiedKFold
import numpy as np
from collections import defaultdict

# Initialize 5-fold cross validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)

# Store results for each fold
fold_results = {
    'accuracies': [],
    'predictions': [],
    'probabilities': [],
    'true_labels': []
}

print("Starting 5-fold cross validation...")
print("-" * 50)

# Perform cross validation
for fold, (train_idx, test_idx) in enumerate(cv.split(data, labels), 1):
    print(f"Fold {fold}/5")
    
    # Split data for current fold
    X_train_fold = [data[i] for i in train_idx]
    X_test_fold = [data[i] for i in test_idx] 
    y_train_fold = [labels[i] for i in train_idx]
    y_test_fold = [labels[i] for i in test_idx]
    
    # Initialize fresh pipeline for each fold
    pipeline = HierarchicalAlzheimersDetectionPipeline(
        word2vec_size=50,
        sentence_lstm_units=64,    # Hidden size for sentence encoding
        session_lstm_units=64,     # Hidden size for session encoding  
        mlp_hidden_units=32
    )
    
    # Train pipeline
    print(f"  Training fold {fold}...")
    train_results = pipeline.fit(X_train_fold, y_train_fold, epochs=25)
    
    # Evaluate on test set
    print(f"  Evaluating fold {fold}...")
    accuracy, predictions, probabilities = pipeline.evaluate(X_test_fold, y_test_fold)
    
    # Store results
    fold_results['accuracies'].append(accuracy)
    fold_results['predictions'].extend(predictions)
    fold_results['probabilities'].extend(probabilities)
    fold_results['true_labels'].extend(y_test_fold)
    
    print(f"  Fold {fold} Accuracy: {accuracy:.4f}")
    print()

# Calculate final cross-validation results
print("=" * 60)
print("FINAL CROSS-VALIDATION RESULTS")
print("=" * 60)

# Overall statistics
mean_accuracy = np.mean(fold_results['accuracies'])
std_accuracy = np.std(fold_results['accuracies'])

print(f"Individual Fold Accuracies: {[f'{acc:.4f}' for acc in fold_results['accuracies']]}")
print(f"Mean CV Accuracy: {mean_accuracy:.4f} ± {std_accuracy:.4f}")
print(f"Min Accuracy: {min(fold_results['accuracies']):.4f}")
print(f"Max Accuracy: {max(fold_results['accuracies']):.4f}")

# Additional metrics if you want to calculate them
from sklearn.metrics import classification_report, confusion_matrix

print("\nOverall Classification Report:")
print(classification_report(fold_results['true_labels'], fold_results['predictions']))

print("\nOverall Confusion Matrix:")
print(confusion_matrix(fold_results['true_labels'], fold_results['predictions']))

# Store final results
final_results = {
    'mean_accuracy': mean_accuracy,
    'std_accuracy': std_accuracy,
    'fold_accuracies': fold_results['accuracies'],
    'all_predictions': fold_results['predictions'],
    'all_probabilities': fold_results['probabilities'],
    'all_true_labels': fold_results['true_labels']
}

print(f"\nCross-validation completed!")
print(f"Final Model Performance: {mean_accuracy:.4f} ± {std_accuracy:.4f}")

Starting 5-fold cross validation...
--------------------------------------------------
Fold 1/5
Using device: cpu
  Training fold 1...
Starting hierarchical pipeline training...
Training Word2Vec model...
Word2Vec vocabulary size: 631
Converting to hierarchical sequences...
Number of sessions: 441
Average sentences per session: 12.2
Average words per sentence: 8.4
Starting training with early stopping...
New best validation accuracy: 56.18% (Epoch 1)
Epoch [1/25]
Train Loss: 0.7407, Train Acc: 49.72%
Val Loss: 0.6924, Val Acc: 56.18%
Patience: 0/7
--------------------------------------------------
Epoch [6/25]
Train Loss: 0.6964, Train Acc: 52.84%
Val Loss: 0.6725, Val Acc: 56.18%
Patience: 5/7
--------------------------------------------------
New best validation accuracy: 60.67% (Epoch 7)
Epoch [7/25]
Train Loss: 0.6545, Train Acc: 59.38%
Val Loss: 0.6176, Val Acc: 60.67%
Patience: 0/7
--------------------------------------------------
New best validation accuracy: 74.16% (Epoch 8)
E

ValueError: Expected more than 1 value per channel when training, got input size torch.Size([1, 32])